# Hướng dẫn chạy Mô hình 3D DATN trên Kaggle qua GitHub
Notebook này tự động thiết lập môi trường ảo Python 3.10, cài đặt các mô hình AI (SDXL, Grounded-SAM2, TRELLIS) và clone mã nguồn của bạn từ GitHub để chạy máy chủ API Ngrok kết nối với Giao diện Web.

## Bước 1: Khởi tạo Python 3.10 và Môi trường ảo (venv)

In [ ]:
import subprocess, os

def run(cmd):
    print(f"Executing: {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.stdout: print(r.stdout.strip())
    if r.stderr and r.returncode != 0: print("ERROR:", r.stderr[-500:])
    return r.returncode

VENV = "/opt/venv310"
PY   = f"{VENV}/bin/python"

print("=== 1. Cài đặt Python 3.10 ===")
run("add-apt-repository ppa:deadsnakes/ppa -y")
run("apt-get update -qq")
run("apt-get install -qq python3.10 python3.10-dev python3.10-venv python3.10-distutils")

print("\n=== 2. Khởi tạo môi trường ảo ===")
run(f"python3.10 -m venv {VENV}")
run(f"{PY} --version")

In [ ]:
!/opt/venv310/bin/pip install -q fastapi uvicorn pyngrok diffusers transformers peft accelerate huggingface_hub spacy trimesh plyfile utils3d pymeshfix pyvista xatlas scipy pillow imageio imageio-ffmpeg opencv-python-headless tqdm easydict rembg[cpu]


## Bước 2: Clone Repo GitHub của bạn và cài đặt thư viện

In [ ]:
# Nhập link Git chứa code của bạn tại đây
GITHUB_REPO = "https://github.com/Tiens0710/DATN-3d.git"  # <-- Thay thế bằng link Repo thật của bạn

import shutil
VENV = "/opt/venv310"
PIP = f"{VENV}/bin/pip"
TEMP_CLONE = "/tmp/my_repo"

# Xoá thư mục cũ nếu có
if os.path.exists(TEMP_CLONE): 
    shutil.rmtree(TEMP_CLONE)

print("=== 1. Tải mã nguồn từ GitHub ===")
!git clone {GITHUB_REPO} {TEMP_CLONE}

# Copy code vào thư mục làm việc chính
if os.path.exists(TEMP_CLONE):
    print("\n=== 2. Đồng bộ mã nguồn vào thư mục làm việc ===")
    !cp -rf {TEMP_CLONE}/* /kaggle/working/
    print("✅ Đã copy files sang /kaggle/working/")
    !ls -l /kaggle/working/
else:
    print("❌ Lỗi: Không thể tải code từ Github. Vui lòng kiểm tra lại link repo.")

## Bước 3: Cài đặt dependencies từ requirements.txt

In [ ]:
print("=== 1. Cài đặt các thư viện từ requirements.txt ===")
# Cài đặt PyTorch và CUDA 12.1 trước để đảm bảo tương thích xformers
!{PIP} install -q torch==2.1.0 torchvision==0.16.0 --index-url https://download.pytorch.org/whl/cu121

# Cài đặt các package khác trong requirements.txt
!{PIP} install -q -r /kaggle/working/requirements.txt

# Ép cài đặt các package bổ sung cần thiết khác
!{PIP} install -q xformers==0.0.22.post7 --index-url https://download.pytorch.org/whl/cu121
!{PIP} install -q spconv-cu121==2.3.8 plyfile==0.9 numpy==1.26.4

print("✅ Cài đặt thư viện hoàn tất!")

## Bước 4: Thiết lập và Biên dịch C++ Extension (nvdiffrast & diff-gaussian-rasterization)

In [ ]:
import os
env = os.environ.copy()
env["CUDA_HOME"] = "/usr/local/cuda"
env["PATH"] = "/usr/local/cuda/bin:" + env.get("PATH", "")

# Chạy trực tiếp qua ký tự ! để hiển thị thời gian thực toàn bộ quá trình biên dịch
!/opt/venv310/bin/pip install -v --no-cache-dir --no-build-isolation --no-deps --force-reinstall git+https://github.com/NVlabs/nvdiffrast.git


In [ ]:
import subprocess, os, shutil

VENV = "/opt/venv310"
PIP  = f"{VENV}/bin/pip"
PY   = f"{VENV}/bin/python"

# 1. Đảm bảo import torch bình thường, nếu không thì tự sửa lỗi numpy
print("=== 1. Kiểm tra môi trường PyTorch ===")
check_torch = subprocess.run([PY, "-c", "import torch; print('PyTorch OK:', torch.__version__)"], capture_output=True, text=True)
if check_torch.returncode != 0:
    print("❌ Lỗi import PyTorch! Đang cài lại numpy...")
    subprocess.run(f"{PIP} install -q --force-reinstall numpy==1.26.4", shell=True)
else:
    print("  ✓ PyTorch hoạt động tốt.")

# 2. Đảm bảo cài đặt wheel và setuptools hỗ trợ bdist_wheel
print("\n=== 2. Kiểm tra bộ cài build-backend (wheel/setuptools) ===")
subprocess.run(f"{PIP} install -q wheel setuptools", shell=True)

# 3. Cài đặt diff-gaussian-rasterization (Mip-Splatting)
print("\n=== 3. Cài đặt diff-gaussian-rasterization ===")
if os.path.exists("/tmp/mip-splatting"):
    shutil.rmtree("/tmp/mip-splatting")
!git clone --recursive https://github.com/autonomousvision/mip-splatting.git /tmp/mip-splatting
!{PIP} install -q --no-build-isolation --no-cache-dir /tmp/mip-splatting/submodules/diff-gaussian-rasterization

# 4. Vá lỗi PyTorch cpp_extension
print("\n=== 4. Vá lỗi PyTorch cpp_extension để bỏ qua check CUDA ===")
cpp_ext_path = f"{VENV}/lib/python3.10/site-packages/torch/utils/cpp_extension.py"
if os.path.exists(cpp_ext_path):
    try:
        with open(cpp_ext_path, "r", encoding="utf-8") as f:
            code = f.read()
        target = "def _check_cuda_version(compiler_name, compiler_version):"
        patched = "def _check_cuda_version(compiler_name, compiler_version):\n    return  # Patched by Antigravity to bypass CUDA version check"
        if target in code and "Patched by Antigravity" not in code:
            code = code.replace(target, patched)
            with open(cpp_ext_path, "w", encoding="utf-8") as f:
                f.write(code)
            print("  ✓ Vá lỗi file cpp_extension.py thành công!")
        else:
            print("  ✓ File cpp_extension.py đã được vá lỗi từ trước.")
    except Exception as e:
        print("  ❌ Lỗi khi vá file cpp_extension.py:", e)

# 5. Biên dịch nvdiffrast
print("\n=== 5. Biên dịch nvdiffrast ===")
env = os.environ.copy()
cuda_path = "/usr/local/cuda"
if os.path.exists(cuda_path):
    env["CUDA_HOME"] = cuda_path
    env["PATH"] = f"{cuda_path}/bin:" + env.get("PATH", "")

print("Đang tiến hành compile nvdiffrast...")
res = subprocess.run(
    f"{PIP} install -v --no-cache-dir --no-build-isolation --no-deps --force-reinstall git+https://github.com/NVlabs/nvdiffrast.git",
    shell=True, env=env, capture_output=True, text=True
)
if res.returncode == 0:
    print("✅ Cài đặt nvdiffrast THÀNH CÔNG!")
else:
    print("❌ Biên dịch lỗi. STDERR:", res.stderr[-500:])


## Bước 5: Cài đặt Repo TRELLIS & Kiểm tra đăng nhập HuggingFace

In [ ]:
# Tải trực tiếp thư mục TRELLIS về thư mục làm việc chính trên Kaggle
!GIT_LFS_SKIP_SMUDGE=1 git clone https://huggingface.co/spaces/trellis-community/TRELLIS /kaggle/working/TRELLIS
print("✅ ĐÃ TẢI XONG THƯ MỤC TRELLIS HỢP LỆ!")


In [ ]:
!/opt/venv310/bin/pip install -q huggingface_hub spacy trimesh

In [ ]:
!/opt/venv310/bin/python -m spacy download en_core_web_sm


In [ ]:
# !/opt/venv310/bin/python /kaggle/working/server.py


In [ ]:
# import time, subprocess
# from pyngrok import ngrok

# # 1. Điền mã ngrok authtoken của bạn tại đây
# NGROK_TOKEN = "3Fj2536cJmPH9jP4D46KqQheptK_2rGfjLmMhfZcZ9UyVKRrB"

# # Thiết lập Token trực tiếp bằng hàm Python (Không dùng lệnh shell)
# ngrok.set_auth_token(NGROK_TOKEN)

# # 2. Khởi động FastAPI server chạy ngầm dưới nền ở cổng 8000
# print("🚀 Đang khởi động FastAPI Server ở cổng 8000...")
# cmd_server = "/opt/venv310/bin/python /kaggle/working/server.py"

# server_proc = subprocess.Popen(cmd_server.split(), stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
# time.sleep(4)

# # 3. Tạo đường truyền công khai (Tunnel) từ Ngrok kết nối với cổng 8000
# try: 
#     ngrok.kill() # Tắt các cổng thừa
#     time.sleep(1)
#     public_url = ngrok.connect(8000)
#     print("\n" + "="*60)
#     print("🎉 KHỞI ĐỘNG HỆ THỐNG THÀNH CÔNG!")
#     print(f"👉 Đường dẫn API Ngrok của bạn: {public_url}")
#     print("="*60 + "\n")
#     print("Hãy copy đường dẫn trên dán vào ô 'Kaggle API URL' ở giao diện index.html trên máy tính của bạn.")
# except Exception as e:
#     print(f"❌ Lỗi khi mở cổng ngrok: {e}")


In [ ]:
# 1. Tải và cài đặt GroundingDINO từ mã nguồn (Mất khoảng 1 phút để biên dịch C++)
!git clone https://github.com/IDEA-Research/GroundingDINO.git /tmp/GroundingDINO
!CUDA_HOME=/usr/local/cuda /opt/venv310/bin/pip install -q -e /tmp/GroundingDINO

# 2. Tải và cài đặt Segment Anything 2 (SAM2)
!git clone https://github.com/facebookresearch/segment-anything-2.git /tmp/segment-anything-2
!/opt/venv310/bin/pip install -q -e /tmp/segment-anything-2

# 3. Tạo thư mục và tải Checkpoint của GroundingDINO
!mkdir -p /kaggle/working/groundingdino_ckpt
!wget -q -O /kaggle/working/groundingdino_ckpt/groundingdino_swint_ogc.pth https://github.com/IDEA-Research/GroundingDINO/releases/download/v0.1.0-alpha/groundingdino_swint_ogc.pth
!cp /tmp/GroundingDINO/groundingdino/config/GroundingDINO_SwinT_OGC.py /kaggle/working/groundingdino_ckpt/

# 4. Tạo thư mục và tải Checkpoint của SAM2
!mkdir -p /kaggle/working/sam2_ckpt
!wget -q -O /kaggle/working/sam2_ckpt/sam2_hiera_small.pt https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_small.pt

print("✅ ĐÃ CÀI ĐẶT VÀ TẢI CHECKPOINTS GROUNDED-SAM2 THÀNH CÔNG!")


## Bước 6: Khởi chạy API Server và tạo link kết nối Ngrok

In [ ]:
!pip install pyngrok

In [ ]:
# 1. Gỡ bỏ hoàn toàn phiên bản cài từ GitHub bị lỗi biên dịch C++
!/opt/venv310/bin/pip uninstall -y groundingdino

# 2. Cài đặt lại phiên bản 'groundingdino-py' sạch từ PyPI như trong Notebook của bạn
!/opt/venv310/bin/pip install --force-reinstall groundingdino-py timm==0.9.16 supervision addict yapf

# 3. Đảm bảo tải tệp config GroundingDINO chính thức từ GitHub
!mkdir -p /kaggle/working/groundingdino_ckpt
!wget -q -O /kaggle/working/groundingdino_ckpt/GroundingDINO_SwinT_OGC.py https://raw.githubusercontent.com/IDEA-Research/GroundingDINO/main/groundingdino/config/GroundingDINO_SwinT_OGC.py

print("✅ ĐÃ KHÔI PHỤC BẢN PYPI VÀ CONFIG THÀNH CÔNG! KHÔNG CÒN BỊ LỖI C++ _C.")


In [ ]:
# 1. Gỡ cài đặt bản lỗi xung đột
!/opt/venv310/bin/pip uninstall -y groundingdino transformers diffusers

# 2. Cài đặt lại đúng phiên bản từ Notebook gốc của bạn
!/opt/venv310/bin/pip install --force-reinstall groundingdino-py timm==0.9.16 supervision addict yapf
!/opt/venv310/bin/pip install --force-reinstall diffusers==0.30.3 transformers==4.41.2 accelerate==0.30.1

print("✅ ĐÃ ĐỒNG BỘ VÀ KHÔI PHỤC MÔI TRƯỜNG GIỐNG 100% NOTEBOOK GỐC!")


In [ ]:
# 1. Đảm bảo setuptools đúng phiên bản hỗ trợ compile C++
!/opt/venv310/bin/pip install -q --force-reinstall setuptools==69.5.1 wheel ninja

# 2. Biên dịch nvdiffrast ép buộc dùng PyTorch 2.1.0 và CUDA hiện tại (không dùng cache cũ)
!CUDA_HOME=/usr/local/cuda PATH=/usr/local/cuda/bin:$PATH /opt/venv310/bin/pip install --force-reinstall --no-deps --no-cache-dir --no-build-isolation git+https://github.com/NVlabs/nvdiffrast.git

print("✅ ĐÃ BIÊN DỊCH VÀ KHÔI PHỤC NVDIFFRAST THÀNH CÔNG!")


In [ ]:
# 1. Tải và cài đặt trực tiếp file biên dịch sẵn của nvdiffrast (bỏ qua bước tự build wheel)
!/opt/venv310/bin/pip install --force-reinstall --no-deps https://github.com/camenduru/wheels/releases/download/colab/nvdiffrast-0.3.1-py3-none-any.whl

# 2. Chạy lệnh kiểm tra import thử trong môi trường ảo
!/opt/venv310/bin/python -c "import nvdiffrast.torch as dr; print('🎉 KẾT QUẢ: NVDIFFRAST IMPORT THÀNH CÔNG!')"


In [ ]:
!/opt/venv310/bin/pip install igraph==0.11.8
# Ép cài đặt lại PyTorch 2.1.0 và xformers đồng bộ từ kho của PyTorch
!/opt/venv310/bin/pip install --force-reinstall torch==2.1.0 torchvision==0.16.0 xformers==0.0.22.post7 --index-url https://download.pytorch.org/whl/cu121



In [ ]:
!/opt/venv310/bin/pip install --force-reinstall numpy==1.26.4


In [ ]:
# Cài đặt chính xác phiên bản commit của utils3d giống trong Notebook của bạn
!/opt/venv310/bin/pip install --force-reinstall --no-build-isolation git+https://github.com/EasternJournalist/utils3d.git@9a4eb15e4021b67b12c460c7057d642626897ec8


In [ ]:
import time, subprocess
from pyngrok import ngrok

# 1. Thay thế mã ngrok authtoken của bạn ở đây
NGROK_TOKEN = "3Fj2536cJmPH9jP4D46KqQheptK_2rGfjLmMhfZcZ9UyVKRrB"  # <-- THAY THẾ BẰNG TOKEN NGROK THẬT CỦA BẠN
!/opt/venv310/bin/ngrok config add-authtoken {NGROK_TOKEN}

# 2. Khởi động FastAPI server chạy ngầm dưới nền ở cổng 8000
print("🚀 Đang khởi động FastAPI Server ở cổng 8000...")
cmd_server = "/opt/venv310/bin/python /kaggle/working/server.py"

server_proc = subprocess.Popen(cmd_server.split(), stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
time.sleep(4)

# 3. Tạo đường truyền công khai (Tunnel) từ Ngrok kết nối với cổng 8000
try: 
    ngrok.kill() # Tắt các cổng thừa
    time.sleep(1)
    public_url = ngrok.connect(8000)
    print("\n" + "="*60)
    print("🎉 KHỞI ĐỘNG HỆ THỐNG THÀNH CÔNG!")
    print(f"👉 Đường dẫn API Ngrok của bạn: {public_url}")
    print("="*60 + "\n")
    print("Hãy copy đường dẫn trên dán vào ô 'Kaggle API URL' ở giao diện index.html trên máy tính của bạn.")
except Exception as e:
    print(f"❌ Lỗi khi mở cổng ngrok: {e}")